In [7]:
import os
import numpy as np
import pandas as pd
import optuna
import xgboost as xgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNetCV, RidgeCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import RobustScaler # Fixed: More resilient to pollution spikes
import warnings

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- CONFIGURATION ---
USE_IMPUTATION = True
WINSORIZE_QUANTILE = 0.99 
LAG_DAYS = [1, 3, 7]
ROLL_WINDOWS = [3, 7]
TARGET_COL = "Number of Admissions"
WIND_DIR_COL = "wind_direction_10m_dominant (°)"
N_SPLITS = 5
N_TRIALS = 25
RANDOM_STATE = 42

# --- 1. FEATURE ENGINEERING ---
def engineer_features(df):
    df = df.copy()
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df = df.sort_values('Timestamp').reset_index(drop=True)
    
    # Cyclical Seasonality
    df["dow_sin"] = np.sin(2 * np.pi * df['Timestamp'].dt.dayofweek / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df['Timestamp'].dt.dayofweek / 7)
    df["month_sin"] = np.sin(2 * np.pi * df['Timestamp'].dt.month / 12)
    df["month_cos"] = np.cos(2 * np.pi * df['Timestamp'].dt.month / 12)
    df["weekend"] = (df['Timestamp'].dt.dayofweek >= 5).astype(int)
    
    # Wind Direction
    if WIND_DIR_COL in df.columns:
        df["wind_sin"] = np.sin(2 * np.pi * df[WIND_DIR_COL] / 360)
        df["wind_cos"] = np.cos(2 * np.pi * df[WIND_DIR_COL] / 360)

    exclude = ['Timestamp', TARGET_COL, WIND_DIR_COL, 'Date']
    weather_cols = [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]
    
    new_feats = {}
    for col in weather_cols:
        for lag in LAG_DAYS:
            new_feats[f"{col}_lag{lag}"] = df[col].shift(lag)
        for roll in ROLL_WINDOWS:
            new_feats[f"{col}_roll{roll}"] = df[col].shift(1).rolling(roll).mean()
            
   
    new_feats["admission_lag7"] = df[TARGET_COL].shift(7)
    
    return pd.concat([df, pd.DataFrame(new_feats)], axis=1).dropna().reset_index(drop=True)

# --- 2. MODEL WRAPPERS ---
def build_keras_model(n_feat, params):
    model = keras.Sequential([
        layers.Input(shape=(n_feat,)),
        *[layers.Dense(params['units'], activation='relu') for _ in range(params['n_layers'])],
        layers.Dropout(params['dropout']),
        layers.Dense(1)
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=params['lr']), loss='mse')
    return model

def get_optuna_params(model_type, X_tr, y_tr, X_va, y_va):
    y_tr, y_va = np.array(y_tr).flatten(), np.array(y_va).flatten()
    def objective(trial):
        if model_type == 'rf':
            p = {'n_estimators': trial.suggest_int("n_estimators", 50, 200), 'max_depth': trial.suggest_int("max_depth", 5, 20)}
            model = RandomForestRegressor(**p, random_state=RANDOM_STATE, n_jobs=-1)
        elif model_type == 'xgb':
            p = {'n_estimators': trial.suggest_int("n_estimators", 50, 300), 'max_depth': trial.suggest_int("max_depth", 3, 10), 'learning_rate': trial.suggest_float("learning_rate", 0.01, 0.2, log=True)}
            model = xgb.XGBRegressor(**p, random_state=RANDOM_STATE)
        elif model_type == 'keras':
            p = {'units': trial.suggest_categorical("units", [32, 64, 128]), 'n_layers': trial.suggest_int("n_layers", 1, 3), 'dropout': trial.suggest_float("dropout", 0.2, 0.5), 'lr': trial.suggest_float("lr", 3e-4, 3e-3, log=True)}
            model = build_keras_model(X_tr.shape[1], p)
            model.fit(X_tr, y_tr, epochs=30, verbose=0)
            return r2_score(y_va, model.predict(X_va).flatten())
        
        model.fit(X_tr, y_tr)
        return r2_score(y_va, model.predict(X_va).flatten())

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS)
    return study.best_params

# --- 3. MAIN EVALUATION LOOP ---
df = engineer_features(pd.read_csv("/Users/suhaniagarwal/Downloads/all_features_data.csv"))
feat_cols = [c for c in df.columns if any(s in c for s in ["_lag", "_roll", "sin", "cos", "weekend"])]
X, y = df[feat_cols], df[TARGET_COL]

tscv = TimeSeriesSplit(n_splits=N_SPLITS)
fold_results = []
best_params = {}

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    if USE_IMPUTATION:
        filler = X_train.median()
        X_train, X_test = X_train.fillna(filler), X_test.fillna(filler)
    
    y_train_capped = y_train.clip(upper=y_train.quantile(WINSORIZE_QUANTILE))
    
    # Fixed: RobustScaler handles pollution outliers better for manuscripts
    scaler = RobustScaler() 
    X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)
    
    if fold == 0:
        print("Tuning models on Fold 1...")
        split = int(len(X_train_s) * 0.8)
        for m in ['rf', 'xgb', 'keras']:
            best_params[m] = get_optuna_params(m, X_train_s[:split], y_train_capped[:split], X_train_s[split:], y_train_capped[split:])

    models = {
        "Dummy (mean)": DummyRegressor(strategy="mean"),
        "Ridge": RidgeCV(alphas=np.logspace(-3, 2, 100), cv=5),
        "RandomForest": RandomForestRegressor(**best_params['rf']),
        "XGBoost": xgb.XGBRegressor(**best_params['xgb']),
        "SVR": SVR(C=1.0, epsilon=0.1)
    }
    
    scores = {}
    for name, model in models.items():
        model.fit(X_train_s, y_train_capped)
        preds = model.predict(X_test_s).flatten()
        
        # Added: Comprehensive metrics for publication
        scores[f"{name}_R2"] = r2_score(y_test, preds)
        scores[f"{name}_MAE"] = mean_absolute_error(y_test, preds)
        scores[f"{name}_RMSE"] = np.sqrt(mean_squared_error(y_test, preds))
    
    # Keras Evaluation
    m_keras = build_keras_model(X_train_s.shape[1], best_params['keras'])
    m_keras.fit(X_train_s, y_train_capped, epochs=100, verbose=0)
    k_preds = m_keras.predict(X_test_s).flatten()
    scores["Keras_R2"] = r2_score(y_test, k_preds)
    scores["Keras_MAE"] = mean_absolute_error(y_test, k_preds)
    scores["Keras_RMSE"] = np.sqrt(mean_squared_error(y_test, k_preds))
    
    fold_results.append(scores)
    print(f"Fold {fold+1} complete.")

# --- 4. FINAL OUTPUT ---
results_summary = pd.DataFrame(fold_results).mean()
print("\nFinal Average Metrics (5-Fold CV):")
print(results_summary)

Tuning models on Fold 1...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
4/4 ━━━━━━━━━━━━━━━━━━━